# 03 — Nested evaluation: every variant × view × label set

For each label set (`L_ctrl` first; then S1: provider, `L_track(1 s)`, `L_track(3 s)`; `L_track(2 s)` ≡ `L_ctrl`) temperature and decoder are re-selected on the six inner validation matches against that label set, and the held-out match is scored once against it. Results already written by notebook 01 (same code path) are reused. Evaluation never fits: a missing prediction is an error.

In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp3_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_3')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp3_common as C
import exp1_common as C1
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp3_protocol as P, exp3_report as R

PREREG verified cff444fff0106e9c | frozen 2026-11-08T16:53:50+00:00


In [2]:
cache = {}
for label_set in R.S1_LABELS:
    for variant in P.VARIANTS:
        for view in C.VIEWS:
            path = C.RESULTS / f'variant_{variant}__{view}__{label_set}.json'
            if path.exists():
                result = C1.read_json(path); tag = 'cached'
            else:
                t0 = time.time()
                result = P.evaluate_variant(variant, view, label_set, cache)
                C1.write_json(path, result); tag = f'{time.time() - t0:.0f}s'
            frame = pd.DataFrame(result['rows'])
            print(f'{label_set:8s} {view:12s} {variant:15s} spellF1 {frame.spell_f1.mean():.4f}  max|cnt| {frame.count_error_ratio.abs().max():.3f}  gamma {sorted(set(frame.gamma))}  ({tag})', flush=True)

ctrl     original     exp3_ctrl_ens3  spellF1 0.7409  max|cnt| 0.175  gamma [0.0, 0.5]  (cached)


ctrl     target_like  exp3_ctrl_ens3  spellF1 0.7418  max|cnt| 0.187  gamma [0.0]  (cached)


ctrl     original     exp3_ref_ens3   spellF1 0.7506  max|cnt| 0.195  gamma [0.0]  (cached)


ctrl     target_like  exp3_ref_ens3   spellF1 0.7483  max|cnt| 0.225  gamma [0.0]  (cached)


ctrl     original     exp3_ctrl       spellF1 0.7283  max|cnt| 0.183  gamma [0.0, 0.5]  (cached)


ctrl     target_like  exp3_ctrl       spellF1 0.7253  max|cnt| 0.194  gamma [0.0, 0.5]  (cached)


... (46 outputs left out)

track_3  original     exp2_aug        spellF1 0.7485  max|cnt| 0.176  gamma [0.0, 0.5]  (cached)


track_3  target_like  exp2_aug        spellF1 0.7515  max|cnt| 0.194  gamma [0.0, 0.5]  (cached)


track_3  original     exp1_ref        spellF1 0.7491  max|cnt| 0.159  gamma [0.0, 0.5]  (cached)


track_3  target_like  exp1_ref        spellF1 0.7444  max|cnt| 0.134  gamma [0.0, 0.5]  (cached)


## Per-match detail on `L_ctrl` (held-out matches)

In [3]:
keep = ['match', 'spell_f1', 'macro_f1', 'n_true', 'n_pred', 'count_error_ratio', 'temperature', 'lambda_team', 'lambda_dead', 'gamma']
for variant in ('exp3_ctrl_ens3', 'exp3_ref_ens3'):
    for view in C.VIEWS:
        frame = pd.DataFrame(R.variant_rows(variant, view, 'ctrl'))
        print(f'== {variant} / {view}'); print(frame[keep].round(3).to_string(index=False))

== exp3_ctrl_ens3 / original
 match  spell_f1  macro_f1  n_true  n_pred  count_error_ratio  temperature  lambda_team  lambda_dead  gamma
J03WMX     0.756     0.930     298     252             -0.154        1.184          4.0          1.0    0.0
J03WN1     0.729     0.909     315     260             -0.175        1.246          4.0          4.0    0.5
J03WOH     0.757     0.931     279     241             -0.136        1.187          1.0          4.0    0.0
J03WOY     0.743     0.920     302     263             -0.129        1.200          2.0          4.0    0.0
J03WPY     0.765     0.932     272     249             -0.085        1.204          2.0          4.0    0.5
J03WQQ     0.748     0.925     284     258             -0.092        1.196          4.0          1.0    0.0
J03WR9     0.689     0.909     267     223             -0.165        1.205          2.0          2.0    0.0
== exp3_ctrl_ens3 / target_like
 match  spell_f1  macro_f1  n_true  n_pred  count_error_ratio  temperature 

## Diagnostic: argmax (no decoder), `L_ctrl`

In [4]:
argmax = pd.concat([pd.DataFrame(P.argmax_rows(v, view, 'ctrl', cache))
                    for v in ('exp3_ctrl_ens3', 'exp3_ref_ens3') for view in C.VIEWS], ignore_index=True)
argmax.to_csv(C.RESULTS / 'argmax_rows.csv', index=False)
print(argmax.groupby(['variant', 'view'])[['spell_f1', 'macro_f1', 'count_error_ratio']].mean().round(4).to_string())

                                   spell_f1  macro_f1  count_error_ratio
variant               view                                              
exp3_ctrl_ens3_argmax original       0.6989    0.9225             0.0546
                      target_like    0.6980    0.9215             0.0532
exp3_ref_ens3_argmax  original       0.6355    0.9203             0.3973
                      target_like    0.6312    0.9192             0.4016


## S2: coverage view (clean ball re-revealed on 12 % of hidden frames), saved weights, fold configs from `target_like`

In [5]:
t0 = time.time()
rows = []
for variant in ('exp3_ctrl_ens3', 'exp3_ref_ens3', 'exp3_ctrl', 'exp2_aug'):
    for label_set in ('ctrl', 'provider'):
        selections = C1.read_json(C.RESULTS / f'variant_{variant}__target_like__{label_set}.json')['selections']
        rows += P.evaluate_coverage(variant, label_set, selections)
coverage = pd.DataFrame(rows)
coverage.to_csv(C.RESULTS / 'coverage_rows.csv', index=False)
print(coverage.groupby(['label_set', 'variant'])[['spell_f1', 'macro_f1', 'count_error_ratio']].mean().round(4).to_string())
print(f'{time.time() - t0:.0f}s')

                          spell_f1  macro_f1  count_error_ratio
label_set variant                                              
ctrl      exp2_aug          0.7464    0.9242            -0.0898
          exp3_ctrl         0.7418    0.9239            -0.1505
          exp3_ctrl_ens3    0.7541    0.9270            -0.1394
          exp3_ref_ens3     0.7552    0.9287            -0.1108
provider  exp2_aug          0.6695    0.9230            -0.1902
          exp3_ctrl         0.5917    0.9139            -0.3669
          exp3_ctrl_ens3    0.6033    0.9180            -0.3672
          exp3_ref_ens3     0.6736    0.9265            -0.1947
40s
